In [2]:
class Economy:
    def __init__(self, YD_1=0, H_1=0):
        self.period = [0]

        #Exogenous inputs, one value stored per period.
        self.G = [0]

        #Computed flows.
        self.Y = [0]
        self.T = [0]
        self.YD = [YD_1]
        self.YD_e = [YD_1]
        self.C = [0]
        self.delta_Hs = [0]
        self.delta_Hh = [0]
        self.delta_Hd = [0]
        self.Hd = [H_1]

        #Stocks (levels carried across periods).
        self.H = [H_1]

    def runPeriod(self, G, alpha1, alpha2, theta):
        YD_1 = self.YD[-1]
        H_1 = self.H[-1]

        #Consumption
        YD_e = YD_1
        C = alpha1 * YD_e + alpha2 * H_1

        #Income
        Y = C + G

        #Taxes and disposable income
        T = theta * Y
        YD = Y - T

        #Flows of funds
        delta_Hs = G - T
        delta_Hh = YD - C

        #Stock update
        H = H_1 + delta_Hs

        #Desired wealth (compared against actual wealth H)
        delta_Hd = YD_e - C
        Hd = H_1 + delta_Hd

        self.period.append(self.period[-1] + 1)
        self.G.append(G)
        self.Y.append(Y)
        self.T.append(T)
        self.YD.append(YD)
        self.YD_e.append(YD_e)
        self.C.append(C)
        self.delta_Hs.append(delta_Hs)
        self.delta_Hh.append(delta_Hh)
        self.H.append(H)
        self.delta_Hd.append(delta_Hd)
        self.Hd.append(Hd)

    def returnCurrentState(self):
        return (self.period[-1], self.G[-1], self.Y[-1], self.T[-1],
                self.YD[-1], self.YD_e[-1], self.C[-1], self.delta_Hs[-1],
                self.delta_Hh[-1], self.H[-1], self.delta_Hd[-1], self.Hd[-1])

    def _print_table(self, indices, title, max_width=76):
        #Same plain style as before; splits into blocks if periods overflow max_width.
        variables = [
            ("G", self.G), ("Y", self.Y), ("T", self.T), ("YD", self.YD),
            ("YD^e", self.YD_e), ("C", self.C), ("dHs", self.delta_Hs),
            ("dHh", self.delta_Hh), ("H", self.H), ("dHd", self.delta_Hd),
            ("Hd", self.Hd),
        ]

        label_width = 8
        col_width = 10

        indices = list(indices)
        cols_per_block = max(1, (max_width - label_width) // col_width)
        blocks = [indices[i:i + cols_per_block]
                  for i in range(0, len(indices), cols_per_block)]

        for block in blocks:
            header_line = f"{'Variable':<{label_width}}" + "".join(
                f"{'Period ' + str(self.period[i]):>{col_width}}" for i in block)
            total_width = len(header_line)

            block_title = title
            if len(blocks) > 1:
                block_title += (f"  (periods {self.period[block[0]]}"
                                 f"-{self.period[block[-1]]})")

            print("=" * total_width)
            print(block_title)
            print("=" * total_width)
            print(header_line)
            print("-" * total_width)

            for name, values in variables:
                row_line = f"{name:<{label_width}}" + "".join(
                    f"{values[i]:>{col_width}.2f}" for i in block)
                print(row_line)

            print("=" * total_width)
            print()

    def printCurrentState(self):
        self._print_table([len(self.period) - 1], "CURRENT STATE \u2014 Model SIM")

    def printHistory(self):
        self._print_table(range(len(self.period)), "SIMULATION RESULTS \u2014 Model SIM")

    def checkConsistency(self, tol=1e-6):
        #Money the government creates must equal money households save.
        title = "STOCK-FLOW CONSISTENCY CHECK (delta_Hh should equal delta_Hs)"
        rows = []
        all_ok = True
        for i in range(1, len(self.period)):
            gap = self.delta_Hh[i] - self.delta_Hs[i]
            ok = abs(gap) < tol
            all_ok = all_ok and ok
            symbol = "\u2713" if ok else "\u2717"
            rows.append(f"Period {self.period[i]:<3} "
                        f"dHh = {self.delta_Hh[i]:>10.2f}   dHs = {self.delta_Hs[i]:>10.2f}   "
                        f"[{symbol}]")
        summary = ("All periods consistent." if all_ok
                   else "Inconsistency detected -- check the equations.")

        inner = max([len(title), len(summary)] + [len(r) for r in rows]) + 2
        top = "\u256d" + "\u2500" * inner + "\u256e"
        mid = "\u251c" + "\u2500" * inner + "\u2524"
        bottom = "\u2570" + "\u2500" * inner + "\u256f"

        print()
        print(top)
        print("\u2502" + title.center(inner) + "\u2502")
        print(mid)
        for row in rows:
            print("\u2502 " + row.ljust(inner - 1) + "\u2502")
        print(mid)
        print("\u2502" + summary.center(inner) + "\u2502")
        print(bottom)


def runEconomy(G, alpha1, alpha2, theta, YD_1, H_1, n):
    econ = Economy(YD_1, H_1)
    for i in range(n):
        econ.runPeriod(G[i], alpha1[i], alpha2[i], theta[i])
    econ.printHistory()
    econ.checkConsistency()
    return econ


econ = runEconomy(
    G=[20, 20, 20],
    alpha1=[0.6, 0.6, 0.6],
    alpha2=[0.4, 0.4, 0.4],
    theta=[0.2, 0.2, 0.2],
    YD_1=0,
    H_1=0,
    n=3
)

SIMULATION RESULTS — Model SIM
Variable  Period 0  Period 1  Period 2  Period 3
------------------------------------------------
G             0.00     20.00     20.00     20.00
Y             0.00     20.00     36.00     48.80
T             0.00      4.00      7.20      9.76
YD            0.00     16.00     28.80     39.04
YD^e          0.00      0.00     16.00     28.80
C             0.00      0.00     16.00     28.80
dHs           0.00     16.00     12.80     10.24
dHh           0.00     16.00     12.80     10.24
H             0.00     16.00     28.80     39.04
dHd           0.00      0.00      0.00     -0.00
Hd            0.00      0.00     16.00     28.80


╭───────────────────────────────────────────────────────────────╮
│ STOCK-FLOW CONSISTENCY CHECK (delta_Hh should equal delta_Hs) │
├───────────────────────────────────────────────────────────────┤
│ Period 1   dHh =      16.00   dHs =      16.00   [✓]          │
│ Period 2   dHh =      12.80   dHs =      12.80   [✓]          │
